In [ ]:
#!pip install -q transformers accelerate bitsandbytes pandas datasets torch

import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
print(f"Устройство: {torch.cuda.get_device_name(0)}")
print(f"Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU доступен: True
Устройство: Tesla T4
Память: 15.6 GB


In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install --upgrade transformers

## Скачивание датасета

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

# Это наш выбранный датасет с каггла, в нем данные сразу разбиты на тест и трейн

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d xuanhuynh233/ielts-dataset
!unzip ielts-dataset.zip

csv_path = 'train_final.csv'
df = pd.read_csv(csv_path)

print(df.head())
print(f"Размер датасета: {len(df)} строк.")

# Удалим строки, где 'essay' или 'band' отсутствуют,
# или 'band' не являетсня числом.

initial_len = len(df)
df = df.dropna(subset=['essay', 'band'])
df = df[pd.to_numeric(df['band'], errors='coerce').notna()]
df['band'] = df['band'].astype(float) # Убедимся, что band это float
print(f"\nУдалено {initial_len - len(df)} строк с отсутствующими или некорректными 'essay'/'band'.")
print(f"Осталось {len(df)} строк после очистки.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: cannot stat '/content/drive/MyDrive/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/xuanhuynh233/ielts-dataset
License(s): MIT
ielts-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  ielts-dataset.zip
replace test_final.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: test_final.csv          
replace train_final.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: train_final.csv         y

                                      band  \
0  7.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
1  5.0\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
2  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
3  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
4    4\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   

              

In [ ]:
df.head()

,band,prompt,essay,TR_Band,TR_Comment,CC_Band,CC_Comment,LR_Band,LR_Mistakes,LR_Corrections,LR_Comment,GRA_Band,GRA_Mistakes,GRA_Corrections,GRA_Comment,Overall_Band,General_Feedback,error
0,7.5,Interviews form the basic criteria for most la...,It is believed by some experts that the tradit...,8.0,The essay sufficiently addresses all parts of ...,8.0,The essay is logically structured and the info...,7.0,exams writing; his teamwork skill is measured;...,written exams; their teamwork skills are measu...,The essay demonstrates a sufficient range of v...,7.0,...his ability to do the work and their proble...,...their ability to do the work and their prob...,"A variety of complex structures is used, and t...",7.5,This is a strong response with a very clear st...,NaN
1,5.0,Interviews form the basic selecting criteria f...,Nowadays numerous huge firms allocate an inter...,5.0,The essay addresses the task only partially. I...,5.0,"The essay is presented with some organization,...",5.0,choosing criteria; up-to-the-minute; reap the ...,selection criterion/criteria; effective/reliab...,The writer attempts to use some less common vo...,5.0,having excellent interpersonal skills are; mak...,having excellent interpersonal skills is; effe...,The essay shows a limited range of sentence st...,5.0,The essay presents a clear position and follow...,NaN
2,5.5,Interview form the basic selection criteria fo...,The interview section is the most vital part o...,6.0,"You have addressed all parts of the question, ...",6.0,The essay has a clear structure with an introd...,5.0,Interview section; prospect for the specified ...,The interview stage; prospective candidate for...,You have used an adequate range of vocabulary ...,5.0,Interview form the basic selection criteria; w...,Interviews form the basic selection criteria; ...,You have attempted to use a mix of simple and ...,5.5,Your essay presents a clear opinion and is log...,NaN
3,5.5,Interviews form the basic selection criteria f...,It is argued that the best method to recruit e...,6.0,The essay addresses the prompt by discussing b...,5.5,"The essay is organized into paragraphs, and th...",5.0,acquired by the recruiter; choosing the best e...,possessed by the applicant; choosing the best ...,The range of vocabulary is limited but minimal...,5.0,who do not lacks in any single question; accor...,who do not lack an answer to any single questi...,The writer attempts to use a mix of simple and...,5.5,Your essay has a clear structure and you prese...,NaN
4,4.0,Interviews from the basic selecting criteria f...,Nowadays many companies conduct interviews bef...,4.0,The response addresses the task only in a mini...,4.0,"The essay presents information and ideas, but ...",4.0,chosing; intetviuvs; campaigns; sending them; ...,choosing; interviews; companies; hiring them; ...,The writer uses only basic and repetitive voca...,4.0,Interviews from the basic selecting criteria; ...,Interviews form the basic selection criteria; ...,Only a very limited range of sentence structur...,4.0,"Your essay attempts to address the prompt, but...",NaN


## Загрузка функций

In [ ]:
# сразу все импорты и загрузки

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr
import re
from tqdm import tqdm
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
# базовый промт
def create_ielts_prompt(essay_to_grade, example_essays):

    prompt = """You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

IMPORTANT:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria
- Provide realistic scores based on IELTS standards

Here are graded examples:

"""

    for i, example in enumerate(example_essays, 1):
        prompt += f"""Example {i}:
Essay: "{example['essay']}..."
TR_Band: {example['TR_Band']}
CC_Band: {example['CC_Band']}
LR_Band: {example['LR_Band']}
GRA_Band: {example['GRA_Band']}
Overall_Band: {example['Overall_Band']}
---

"""

    prompt += f"""Now, grade this essay following IELTS standards:

Essay: "{essay_to_grade}"

Provide your evaluation in this EXACT format:
TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X
Explanation: [Brief justification for each criterion]
"""

    return prompt


In [ ]:
# получим все оценки из ответа модели, чтобы считать по ним метрики качества

def extract_ielts_scores(response):

    # Извлечем 5 оценок {'TR': float, 'CC': float, 'LR': float, 'GRA': float, 'Overall': float}

    scores = {}

    patterns = {
        'TR': r'TR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'CC': r'CC[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'LR': r'LR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'GRA': r'GRA[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'Overall': r'Overall[_\s]*Band:\s*(\d+(?:\.\d)?)'
    }

    for criterion, pattern in patterns.items():
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            score = float(match.group(1))
            score = round(score * 2) / 2
            scores[criterion] = min(max(score, 0), 9)
        else:
            scores[criterion] = None

    if scores['Overall'] is None and all(scores[k] is not None for k in ['TR', 'CC', 'LR', 'GRA']):
        avg = np.mean([scores['TR'], scores['CC'], scores['LR'], scores['GRA']])
        scores['Overall'] = round(avg * 2) / 2

    return scores

In [ ]:
# функция оценки эссе по нашему промту
def grade_ielts_essay(essay, example_essays, model, tokenizer, max_tokens=600):


    prompt = create_ielts_prompt(essay, example_essays)
    # задаем роль
    messages = [
        {"role": "system", "content": "You are an expert IELTS Writing examiner with years of experience."},
        {"role": "user", "content": prompt}
    ]


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.2,  # Очень низкая для стабильности, была 0.3, выдавала разницу в качестве в 1 единцу MAE
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    return response


In [ ]:
# оцениваем качество модели

def calculate_ielts_metrics(actual, predicted):

    mask = ~(pd.isna(actual) | pd.isna(predicted))
    actual = np.array(actual)[mask]
    predicted = np.array(predicted)[mask]

    if len(actual) == 0:
        return None

    # MAE
    mae = mean_absolute_error(actual, predicted)

    # RMSE
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    # MBE чтобы понимать занижение/завышение оценок
    mbe = np.mean(predicted - actual)

    # Корреляция Спирмена
    correlation, _ = spearmanr(actual, predicted)

    # Точность
    acc_05 = np.mean(np.abs(predicted - actual) <= 0.5)
    acc_10 = np.mean(np.abs(predicted - actual) <= 1.0)

    # Процент точных совпадений (особенно важно для 0.5 шага)
    exact_match = np.mean(predicted == actual)

    # QWK
    from sklearn.metrics import cohen_kappa_score
    qwk = cohen_kappa_score(
        (actual * 2).astype(int),  # Переводим в целые (умножаем на 2)
        (predicted * 2).astype(int),
        weights='quadratic'
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'MBE': mbe,
        'Correlation': correlation,
        'QWK': qwk,
        'Accuracy_±0.5': acc_05,
        'Accuracy_±1.0': acc_10,
        'Exact_Match': exact_match,
        'n_samples': len(actual)
    }

## Сравнение разных моделей

In [ ]:
#!pip uninstall -y bitsandbytes
#!pip install bitsandbytes>=0.46.1

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2


In [ ]:
# перечисляю модели для сравнения
model_names = [
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "nvidia/Gemma-4-31B-IT-NVFP4"
]

sample_size = 30
test_df = df.sample(sample_size, random_state=42).copy()

predictions = {
    'Model_name': [],
    'TR': [],
    'CC': [],
    'LR': [],
    'GRA': [],
    'Overall': []
  }

models_time = {
    'Model_name': [],
    'Model_load_time (sec)':[],
    'Model_load_memory (MB)':[],
    'Avg_inference_time (sec)': [],
    'Total_inference_time (sec)': []
}

# будем давать 5 примеров оценненых эссе внутри промта
examples = df[~df.index.isin(test_df.index)].sample(5).to_dict('records')

print(f"Оценка {sample_size} IELTS эссе...")
print()

for current_model_name in model_names:

  explanations = []
  print(f'Начало загрузки модели {current_model_name}')

  # Конфигурация 4-bit квантизации - чтобы быстро скачать и чтобы быстро работало
  bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
  )

  # Запоминаю память и время на момент старта загрузки модели
  start_time = time.time()
  start_memory = torch.cuda.memory_allocated() if torch.cuda.is_available() else psutil.virtual_memory().used
  current_model = AutoModelForCausalLM.from_pretrained(
    current_model_name,
    quantization_config=bnb_config,
    device_map="auto"
  )

  tokenizer = AutoTokenizer.from_pretrained(current_model_name)

  # Запоминаю память и время на момент окончания загрузки модели
  load_time = time.time() - start_time
  end_memory = torch.cuda.memory_allocated() if torch.cuda.is_available() else psutil.virtual_memory().usedн
  memory_used_mb = (end_memory - start_memory) / (1024 ** 2)
  models_time['Model_name'].append(current_model_name)
  models_time['Model_load_time (sec)'].append(load_time)
  models_time['Model_load_memory (MB)'].append(memory_used_mb)
  print(f'Окончание загрузки модели {current_model}')

  inference_times = []
  for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
      start_infer = time.time()
      predictions['Model_name'] = current_model_name
      try:
          # Генерация оценки моделью по промту
          response = grade_ielts_essay(
              row['essay'],
              examples,
              current_model,
              tokenizer,
              max_tokens=600
          )

          # Извлечение оценок из ответа модели
          scores = extract_ielts_scores(response)

          for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
              predictions[criterion].append(scores[criterion])

          explanations.append(response)

      except Exception as e:
          print(f"\n Ошибка на эссе {idx}: {e}")
          for criterion in predictions.keys():
              predictions[criterion].append(None)
          explanations.append("")

  # Фиксирую время
  inference_times.append(time.time() - start_infer)
  avg_inference_time = np.mean(inference_times)
  sum_inference_time = np.sum(inference_times)
  models_time['Avg_inference_time (sec)'].append(avg_inference_time)
  models_time['Total_inference_time (sec)'].append(sum_inference_time)
  test_df[f'pred_model'] = predictions['Model_name']
  # Добавляем результаты в датафрейм
  for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
      test_df[f'pred_{criterion}'] = predictions[criterion]

  test_df['explanation'] = explanations

  print("\n Оценка завершена!")


Оценка 30 IELTS эссе...

Начало загрузки модели Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Окончание загрузки модели Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    


100%|██████████| 30/30 [24:28<00:00, 48.95s/it]



 Оценка завершена!
Начало загрузки модели Qwen/Qwen2.5-7B-Instruct


...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Окончание загрузки модели Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    



  0%|          | 0/30 [00:00<?, ?it/s]

  3%|▎         | 1/30 [00:48<23:24, 48.42s/it]

  7%|▋         | 2/30 [01:47<25:25, 54.49s/it]

 10%|█         | 3/30 [02:44<25:09, 55.90s/it]

 13%|█▎        | 4/30 [03:29<22:17, 51.43s/it]

 17%|█▋        | 5/30 [04:26<22:21, 53.67s/it]

 20%|██        | 6/30 [05:19<21:17, 53.21s/it]

 23%|██▎       | 7/30 [06:01<18:57, 49.46s/it]

 27%|██▋       | 8/30 [06:57<18:53, 51.54s/it]

 30%|███       | 9/30 [07:44<17:36, 50.31s/it]

 33%|███▎      | 10/30 [08:53<18:41, 56.05s/it]

 37%|███▋      | 11/30 [09:45<17:21, 54.79s/it]

 40%|████      | 12/30 [10:30<15:32, 51.81s/it]

 43%|████▎     | 13/30 [11:22<14:43, 51.97s/it]

 47%|████▋     | 14/30 [12:20<14:17, 53.62s/it]

 50%|█████     | 15/30 [13:13<13:24, 53.60s/it]

 53%|█████▎    | 16/30 [14:12<12:50, 55.07s/it]

 57%|█████▋    | 17/30 [15:13<12:21, 57.05s/it]

 60%|██████    | 18/30 [16:15<11:42, 58.54s/it]

 63%|██████▎   | 19/30 [17:14<10:44, 58.60s/it]

 67%|██████▋   | 20/30 [18:12<09:43,

ValueError: Length of values (60) does not match length of index (30)

In [ ]:
# Датасет с временем работы моделей
models_time_df = pd.DataFrame(models_time)
models_time_df = models_time_df.round(3)
models_time_df

In [ ]:
# Датасет с эссе и предсказаниями моделей по ним
test_df.head()

In [ ]:
print("Считаю метрики качества")

criteria_mapping = {
    'TR': 'TR_Band',
    'CC': 'CC_Band',
    'LR': 'LR_Band',
    'GRA': 'GRA_Band',
    'Overall': 'Overall_Band'
}

all_metrics = {'Criterion':[], 'Model_name':[],'MAE':[], 'RMSE':[], 'MBE':[]
               , 'Correlation':[], 'QWK':[], 'Accuracy_±0.5':[], 'Accuracy_±1.0':[], 'Exact_Match':[], 'n_samples':[]}

for current_model_name in model_names:
  #print(current_model_name)
  test_df_model = test_df.loc[test_df['pred_model'] == current_model_name]
  for criterion, actual_col in criteria_mapping.items():
      actual = test_df_model[actual_col]
      predicted = test_df_model[f'pred_{criterion}']

      metrics = calculate_ielts_metrics(actual, predicted)

      if metrics:
        all_metrics['Criterion'].append(criterion)
        all_metrics['Model_name'].append(current_model_name)
        all_metrics['MAE'].append(round(metrics['MAE'], 3))
        all_metrics['RMSE'].append(round(metrics['RMSE'], 3))
        all_metrics['MBE'].append(round(metrics['MBE'], 3))
        all_metrics['Correlation'].append(round(metrics['Correlation'], 3))
        all_metrics['QWK'].append(round(metrics['QWK'], 3))
        all_metrics['Accuracy_±0.5'].append(round(metrics['Accuracy_±0.5'], 3))
        all_metrics['Accuracy_±1.0'].append(round(metrics['Accuracy_±1.0'], 3))
        all_metrics['Exact_Match'].append(round(metrics['Exact_Match'], 3))
        all_metrics['n_samples'].append(round(metrics['n_samples'], 3))

df_all_metrics = pd.DataFrame(all_metrics)
df_all_metrics

In [ ]:
# финальный датасет
df_comparison = pd.merge(df_all_metrics, models_time_df, left_on = 'Model_name', right_on = 'Model_name', how = 'inner')
df_comparison

## Финальные датасеты со сравнениями

In [ ]:
# финальное сравнение моделей по Overall
df_comparison.loc[df_comparison['Criterion'] == 'Overall']

In [ ]:
# финальное сравнение моделей по TR
df_comparison.loc[df_comparison['Criterion'] == 'TR']

In [ ]:
# финальное сравнение моделей по CC
df_comparison.loc[df_comparison['Criterion'] == 'CC']

In [ ]:
# финальное сравнение моделей по LR
df_comparison.loc[df_comparison['Criterion'] == 'LR']

In [ ]:
# финальное сравнение моделей по GRA
df_comparison.loc[df_comparison['Criterion'] == 'GRA']